In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

search_tool = SerperDevTool()   
scrape_tool = ScrapeWebsiteTool()
print("✅ Tools initialized")

fast_llm = LLM(
    model="groq/llama-3.1-8b-instant",
    api_base="https://api.groq.com/openai/v1  ",
    api_key=os.environ["GROQ_API_KEY"],
    provider="groq",
    temperature=0.7,
    max_tokens=1024
)

smart_llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    api_base="https://api.groq.com/openai/v1  ",
    api_key=os.environ["GROQ_API_KEY"],
    provider="groq",
    temperature=0.7,
    max_tokens=1500
)
print("✅ LLMs initialized")


4: Create Agents
market_researcher = Agent(
    role="Senior Market Researcher",
    goal="Gather comprehensive, accurate market data and industry trends",
    backstory="You are a senior market researcher with 15 years of experience in technology markets. You are known for thorough data collection and always citing your sources. You focus on quantitative data: market size, growth rates, funding rounds, and user statistics.",
    tools=[search_tool,scrape_tool],
    llm=fast_llm,
    verbose=True
)

industry_analyst = Agent(
    role="Industry Analyst",
    goal="Identify actionable insights, opportunities, and threats from market data",
    backstory="You are a strategic analyst at a top consulting firm. You excel at finding patterns in data, identifying market gaps, and assessing competitive dynamics. You always support your conclusions with evidence from the research.",
    llm=fast_llm,
    verbose=True
)

report_writer = Agent(
    role="Executive Report Writer",
    goal="Produce clear, professional market research reports",
    backstory="You are a business writer who specializes in executive-level reports. You structure information for quick comprehension, use clear headings and bullet points, and always include an executive summary. You write for busy decision-makers.",
    llm=smart_llm,
    verbose=True
)



# CELL 5: Create Tasks
research_task = Task(
    description=(
        "Conduct thorough market research on the {industry} industry. "
        "Your research must cover:\n"
        "1. Current market size (in USD) and projected growth rate\n"
        "2. Top 5 companies by market share with brief descriptions\n"
        "3. Recent trends shaping the industry (last 12 months)\n"
        "4. Key technologies driving change\n"
        "5. Recent notable funding rounds or acquisitions\n\n"
        "Focus on data from 2024-2025. Cite sources where possible."
    ),
    expected_output=(
        "A detailed research document with sections for market size, "
        "key players, trends, technologies, and recent deals. "
        "Include specific numbers and data points."
    ),
    agent=market_researcher
)

analysis_task = Task(
    description=(
        "Analyze the market research on the {industry} industry and provide "
        "strategic insights. Your analysis must include:\n"
        "1. SWOT analysis of the overall market\n"
        "2. Identification of the top 3 market opportunities\n"
        "3. Assessment of the top 3 market threats\n"
        "4. Competitive landscape analysis — who is winning and why\n"
        "5. Prediction for the next 2-3 years\n\n"
        "Support every conclusion with data from the research."
    ),
    expected_output=(
        "A strategic analysis document with SWOT analysis, "
        "ranked opportunities and threats, competitive assessment, "
        "and market predictions, all supported by research data."
    ),
    agent=industry_analyst,
    context=[research_task]
)

report_task = Task(
    description=(
        "Write a professional market research report on the {industry} industry "
        "based on the research and analysis provided. The report must include:\n"
        "1. Executive Summary (150 words max)\n"
        "2. Market Overview (size, growth, key statistics)\n"
        "3. Competitive Landscape (top players, market share)\n"
        "4. Trends and Opportunities\n"
        "5. Risks and Challenges\n"
        "6. Strategic Recommendations (3-5 actionable items)\n\n"
        "Format: Use Markdown with clear headings. Keep the total length "
        "between 1000-1500 words. Write for a C-level audience."
    ),
    expected_output=(
        "A polished market research report in Markdown format, "
        "1000-1500 words, with executive summary, market overview, "
        "competitive landscape, trends, risks, and recommendations."
    ),
    agent=report_writer,
    context=[research_task, analysis_task],
    output_file="output/market_research_report.md"
)



# CELL 6: Create Crew
market_crew = Crew(
    agents=[market_researcher, industry_analyst, report_writer],
    tasks=[research_task, analysis_task, report_task],
    process=Process.sequential,
    verbose=True
)


 CELL 7: Create output directory and run
# CELL 7: Create output directory and run
import os
os.makedirs("output", exist_ok=True)

result = market_crew.kickoff(inputs={
    "industry": "AI code assistants"
})

print("=" * 50)
print("FINAL REPORT")
print("=" * 50)
print(result.raw)